# `NSE` json tests

In [ ]:
# hack to change jupyter directory in notebooks for imports
import os
from pathlib import Path
if Path.cwd().parts[-1] == 'notebooks':
    root = Path.cwd().parent
else:
    root = Path.cwd()
os.chdir(root)

# Logger
import logging, logging.config
import yaml

with open(root / 'config' / 'log.yml', 'r') as f:
    config = yaml.safe_load(f.read())
    logging.config.dictConfig(config)

log = logging.getLogger('ib_log')

In [ ]:
from src.nse.nse import nse_json
import pandas as pd
import numpy as np

symbol = "INFY"

In [ ]:
# testing equity price watch url

url = "https://www1.nseindia.com/live_market/dynaContent/live_watch/stock_watch/foSecStockWatch.json"

js = nse_json(url)
x = []

for item in js['data']:
    df = pd.DataFrame.from_dict([item])
    x.append(df)
df = pd.concat(x, ignore_index=True)

df.head()

In [35]:
# testing an index price watch url

url = 'https://iislliveblob.niftyindices.com/jsonfiles/LiveIndicesWatch.json'
js = nse_json(url) # works!

x = []
for item in js['data']:
    df = pd.DataFrame.from_dict([item])
    x.append(df)
df = pd.concat(x, ignore_index=True)

# keep only equities
df = df[(df.indexType == 'eq') & (df.yearHigh != '-')].reset_index(drop=True)

# symbol map
di = {'NIFTY 50': 'NIFTY50', 'NIFTY BANK': 'BANKNIFTY', 'INDIA VIX': 'INDIAVIX', 'NIFTY IT': 'NIFTYIT'}
df.insert(0, 'symbol', df.indexName.map(di).fillna(df.indexName))

In [38]:
# testing a plan marketStatus url
url = 'https://nseindia.com/api/marketStatus'
js = nse_json(url) # works!
nse_is_open = js[list(js.keys())[0]][0]['marketStatus'] != 'Closed'
nse_is_open

True

In [ ]:
# testing a url with a csv file
# similar to src/nse/nse.py/get_nse_syms()

symbol = "" # Empty means all symbols

url = "https://archives.nseindia.com/content/fo/fo_mktlots.csv"
js = nse_json(url)

df = pd.read_csv(url)
df.columns = df.columns.str.strip().str.lower() # cleanup column names

# strip all string contents of whitespaces
df = df.applymap(lambda x: x.strip()
         if type(x) is str else x)

# remove 'Symbol' row
df = df[df.symbol != "Symbol"]

# introduce `secType`
df.insert(1, 'secType', np.where(df.symbol.str.contains('NIFTY'), 'IND', 'STK'))

# introduce `exchange`
df.insert(2, 'exchange', 'NSE')

if symbol:
    result = df[(df.iloc[:, 1] == symbol.upper())]
else:
    result = df
